# Быстрая проверка решения

Для запуска нужны три исходных Parquet в `data/` и ядро **Avito ranking**.
Выберите **Run → Run All Cells**. Обучение здесь не повторяется.

Ноутбук заново строит обучающую историю и поисковые индексы, пересчитывает все 2 452 запроса и сравнивает ответ с приложенным CSV. Установка окружения описана в [README](../README.md).

In [1]:
import json
import os
from pathlib import Path
import subprocess
import sys
import time

os.environ.update({"POLARS_MAX_THREADS": "2", "OPENBLAS_NUM_THREADS": "2", "OMP_NUM_THREADS": "2"})

from avito_ranker.config import load_config
from avito_improved.run import run_stage

project_dir = Path.cwd().resolve()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent
config_path = project_dir / "configs/improved.toml"
config = load_config(config_path)
work = config["work_dir"] / "quality"
saved = config["results_dir"]
started = time.perf_counter()
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               cwd=project_dir, check=True)

CompletedProcess(args=['D:\\Codex_artifacts\\Avito_test\\quality_verification\\environment\\Scripts\\python.exe', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], returncode=0)

## 1. Подготовка

История содержит только обучающие группы. Она связывает похожие запросы с выбранными объявлениями и подкатегориями услуг. Исходные данные проверяются до расчёта.

In [2]:
run_stage("prepare", config_path)
run_stage("restore", config_path)

Начат этап: prepare


resource module not available on Windows
Источник обработан: benchmark_items.parquet осталось items: 326683
Источник обработан: train.parquet осталось items: 0
{'corpus_items': 515895, 'corpus_rule': 'all unique items; benchmark features take precedence on duplicates', 'strict_group': 'sorted unique Snowball tokens; punctuation/case/word order ignored', 'groups_disjoint': True, 'holdout_previously_seen_groups': 0, 'query_selection_uses_labels': False, 'corpus_selection_uses_labels': False, 'query_counts': {'train': 8000, 'dev': 2000, 'holdout': 2000}, 'all_group_counts': [{'fold': 'dev', 'len': 11217}, {'fold': 'holdout', 'len': 6349}, {'fold': 'train', 'len': 50737}], 'sources': {'train.parquet': 'e150ab7a5c98769f643b95ee3a02dc0f664b647facd69a7d280ec6d48ca554a7', 'benchmark_queries.parquet': 'e49de4fb76f03979a4af96034817767d39ae5de9f94e254fed028243188dda06', 'benchmark_items.parquet': '193b3a3961464620cbf8d8797152b7f90cae1826ca749fb7817a210984a09899'}, 'split_hashes': {'dev_queries.pa

Начат этап: restore


resource module not available on Windows
Модель и контрольные суммы проверены



## 2. Поисковые индексы

Строятся BM25-индексы заголовков, описаний и параметров. Итоговый поиск использует только объявления из benchmark.

In [3]:
run_stage("benchmark_indices", config_path)

Начат этап: benchmark_indices


Начат этап: benchmark_index_title
resource module not available on Windows
Tokenized title stemming True vocab 26204 tokens 869708 seconds 10.1
INDEX DONE {"field": "title", "stemming": true, "documents": 189212, "vocabulary": 26205, "tokens": 869708, "nnz": 851848, "seconds": 14.486721515655518, "k1": 1.5, "b": 0.75, "method": "lucene", "corpus_manifest_sha256": "b124e81efa4cfa951bfe8390f371098ea9808621105e016d26bffcddc9fb3822", "corpus_sha256": "193b3a3961464620cbf8d8797152b7f90cae1826ca749fb7817a210984a09899", "normalization": "NFKC lower ё→е; Unicode alphanumeric tokens; no stopword removal"}

Начат этап: benchmark_index_params
resource module not available on Windows
Tokenized params stemming True vocab 78736 tokens 27714318 seconds 57.7
INDEX DONE {"field": "params", "stemming": true, "documents": 189212, "vocabulary": 78737, "tokens": 27714318, "nnz": 11675448, "seconds": 78.07732963562012, "k1": 1.5, "b": 0.75, "method": "lucene", "corpus_manifest_sha256": "b124e81efa4cfa951bfe

## 3. Кандидаты и признаки

К текстовому поиску добавляются кандидаты по фильтрам и похожим запросам из train. Модель учитывает текст, локацию, расстояние, фильтры и свойства объявления.

In [4]:
run_stage("benchmark_features", config_path)

Начат этап: benchmark_features


resource module not available on Windows
benchmark 200 13.9
benchmark 400 27.0
benchmark 600 40.1
benchmark 800 52.9
benchmark 1000 65.8
benchmark 1200 79.0
benchmark 1400 92.0
benchmark 1600 105.1
benchmark 1800 118.2
benchmark 2000 131.0
benchmark 2200 143.9
benchmark 2400 156.9
{
  "fold": "benchmark",
  "queries": 2452,
  "seconds": 160.47172570001567,
  "pool_recall": null,
  "average_pool_size": 1265.5513866231647
}



## 4. Ответ

Сохранённая модель выбирает до 50 объявлений для каждого запроса. Проверяются исходные идентификаторы и формат файла.

In [5]:
run_stage("predict", config_path)

Начат этап: predict


resource module not available on Windows
{
  "valid": true,
  "rows": 2452,
  "min_items": 50,
  "max_items": 50,
  "sha256": "f4c3bbfda1f83fdf1ebe84b01da6afe7e0436ba6f1016bb4557c22cab3556feb",
  "model_sha256": "3107ea0b82c189e86e1ee524e6fd65d9b8cb1430aa2a269054c7d0060628ccee"
}



## 5. Сравнение с готовым файлом

Успешная проверка заканчивается `status: passed` и `answer_identical: true`.

In [6]:
from avito_retrieval.submission import validate_answer

computed = work / "answer.csv"
reference = project_dir / "answer.csv"
report = validate_answer(computed, config["data_dir"] / "benchmark_queries.parquet",
                         config["data_dir"] / "benchmark_items.parquet")
if computed.read_bytes() != reference.read_bytes():
    raise AssertionError("Новый answer.csv отличается от приложенного")
report.update({"status": "passed", "answer_identical": True,
               "seconds": round(time.perf_counter() - started, 2)})
(work / "check_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
report

resource module not available on Windows


{'valid': True,
 'rows': 2452,
 'min_items': 50,
 'max_items': 50,
 'sha256': 'f4c3bbfda1f83fdf1ebe84b01da6afe7e0436ba6f1016bb4557c22cab3556feb',
 'status': 'passed',
 'answer_identical': True,
 'seconds': 501.32}